# MaiBhoj-NLP — Maithili Headline, ZERO-SHOT, all models

One notebook for all six systems. Set `MODEL_KEY` in Cell 1, Run All, repeat.

`MODEL_KEY` options: `airavata` · `sarvam1` · `openhathi` · `llama31` · `gemma2` · `indicbart_pre`

## What changed from your one-shot notebooks

| | one-shot version | this version | why |
|---|---|---|---|
| Exemplar | `USE_FEWSHOT=True`, gold summary drawn from the scored dataframe | **removed entirely** | zero-shot, and it was test-set leakage |
| Eval set | full 2,010 rows | **201-row test split** | the only rows fine-tuned IndicBART can be scored on |
| `MAX_NEW_TOKENS` | 50 | **64** | matches fine-tuned IndicBART |
| Decoding | greedy (`num_beams=1`) | **beams=4, len_pen=1.0, no_repeat=3** | matches both IndicBART variants |
| Precision | 4-bit, **except Sarvam-1 at bf16** | **4-bit for all** | Sarvam-1 had an accidental advantage |
| Headline post-processing | first line, first sentence, **hard 10-word cap** | **cleanup only, no cap** | the cap produced Table 9's length gap |
| Empty output | dropped from its own average | **scored as 0** | dropping inflated baseline means |
| BLEU | corpus-level, `intl` | **sentence-level, smoothed, `intl`** | consistent + per-item for CIs |
| CIs | none | **bootstrap 95% on every metric** | Reviewer 3's open request |

**Prompt is unchanged and verbatim.** Your five one-shot notebooks used a byte-identical prompt
(sha1 `bfcb7cab8511df2a`); it is reproduced here exactly, minus the exemplar block. The
affect-preservation clauses are retained.

**Consistency with the fine-tuned notebooks:** same test split (seed 42), same `headline` reference
column (`article_headline` renamed, as in `maithili_finetune_indicBART`), same decoding, same
evaluator. Fine-tuned IndicBART receives no prompt by design — it is trained, not instructed.

## Before running
1. `python make_test_splits.py` → must print `mai: total=2010 train=1608 val=201 test=201`
2. `mai_test.csv` present in the working directory
3. `export HF_TOKEN=...` (Llama-3.1 and Gemma-2 are gated)


In [1]:
# ============================================================
# CELL 0 — environment fixes. MUST RUN FIRST, BEFORE ANY IMPORT.
# ------------------------------------------------------------
# libcuda.so.1 lives in /lib64 on this cluster but LIBRARY_PATH was unset,
# so gcc could not link it and Triton's JIT build failed. LIBRARY_PATH is
# the link-time path; LD_LIBRARY_PATH is the runtime one.
# Triton is also disabled: beam-search generation does not need it, and it
# was only being invoked by bitsandbytes probing during the 4-bit load.
# ============================================================
import os

for var, path in (("LIBRARY_PATH", "/lib64"), ("LD_LIBRARY_PATH", "/lib64")):
    cur = os.environ.get(var, "")
    if path not in cur.split(":"):
        os.environ[var] = f"{path}:{cur}" if cur else path

os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"]   = "1"

print("LIBRARY_PATH   :", os.environ["LIBRARY_PATH"])
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])
print("Triton JIT     : disabled")
print("\nIf you re-run after a Triton failure, clear its stale cache:")
print("  !rm -rf ~/.triton/cache")


LIBRARY_PATH   : /lib64
LD_LIBRARY_PATH: /lib64
Triton JIT     : disabled

If you re-run after a Triton failure, clear its stale cache:
  !rm -rf ~/.triton/cache


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
# ============================================================
# CELL 1 — pick the model. Nothing else needs editing.
# ============================================================
MODEL_KEY = "indicbart_pre"   # gemma2 (ready) | airavata | sarvam1 | openhathi | indicbart_pre
                        # llama31 -> still pending Meta approval

LANG, TASK = "mai", "headline"

# ---- fixed protocol (identical for every model) ----
SHOTS            = 0
NUM_BEAMS        = 4
LENGTH_PENALTY   = 1.0
NO_REPEAT_NGRAM  = 3
DO_SAMPLE        = False
MAX_NEW_TOKENS   = 64
MAX_INPUT_TOKENS = 1024
USE_4BIT         = True
BATCH_SIZE       = 4
SEED             = 42
FRESH_START      = True

REGISTRY = {
    "airavata":      ("ai4bharat/Airavata",                   "causal", "airavata"),
    "sarvam1":       ("sarvamai/sarvam-1",                    "causal", "plain"),
    "openhathi":     ("sarvamai/OpenHathi-7B-Hi-v0.1-Base",   "causal", "plain"),
    "llama31":       ("meta-llama/Llama-3.1-8B-Instruct",     "causal", "chat_sys"),
    "gemma2":        ("google/gemma-2-9b-it",                 "causal", "chat_nosys"),
    "indicbart_pre": ("ai4bharat/IndicBART",                  "seq2seq","none"),
}
MODEL_NAME, ARCH, FMT = REGISTRY[MODEL_KEY]

TEST_CSV = "mai_test.csv"
RUN_TAG        = f"{MODEL_KEY}_{LANG}_{TASK}_zeroshot"
CHECKPOINT_CSV = f"ckpt_{RUN_TAG}.csv"
FINAL_CSV      = f"gen_{RUN_TAG}.csv"
PERITEM_CSV    = f"per_item_{RUN_TAG}.csv"
METRICS_JSON   = f"metrics_{RUN_TAG}.json"
MANIFEST_JSON  = f"manifest_{RUN_TAG}.json"

print(f"RUN {RUN_TAG}\n  model  = {MODEL_NAME}\n  arch   = {ARCH}  fmt = {FMT}")
print(f"  beams={NUM_BEAMS} max_new={MAX_NEW_TOKENS} 4bit={USE_4BIT} shots={SHOTS}")


RUN indicbart_pre_mai_headline_zeroshot
  model  = ai4bharat/IndicBART
  arch   = seq2seq  fmt = none
  beams=4 max_new=64 4bit=True shots=0


In [4]:
# ============================================================
# CELL 2 — environment, seeding, GPU guard
# ============================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"]      = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

import re, gc, json, time, random, unicodedata
import numpy as np, pandas as pd, torch

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("torch", torch.__version__, "| cuda", torch.version.cuda,
      "| available", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. A silent CPU fallback will take days — fix the kernel before continuing."
)
print("device:", torch.cuda.get_device_name(0),
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

if MODEL_KEY in ("llama31", "gemma2"):
    from huggingface_hub import login
    tok = os.environ.get("HF_TOKEN")
    if tok:
        login(tok); print("HF login OK")
    else:
        print("WARNING: HF_TOKEN not set —", MODEL_NAME, "is gated and will fail to download.")


torch 2.14.0+cu126 | cuda 12.6 | available True
device: NVIDIA RTX A5000 | 25 GB


In [5]:
# ============================================================
# CELL 2b — Hugging Face auth (gated models only)
# ------------------------------------------------------------
# Reads the token from the environment so it is NEVER saved in the
# notebook. Before launching Jupyter:
#       export HF_TOKEN=hf_xxxxxxxxxxxxxxxxx
# or set it here at runtime with getpass (not stored in the file).
# ============================================================
if MODEL_KEY in ("llama31", "gemma2"):
    from huggingface_hub import login, whoami
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("HF token (input hidden): ").strip()
        os.environ["HF_TOKEN"] = tok
    login(tok, add_to_git_credential=False)
    print("logged in as:", whoami()["name"])

    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(MODEL_NAME, "config.json")
        print(f"access to {MODEL_NAME}: OK")
    except Exception as e:
        raise SystemExit(
            f"No access to {MODEL_NAME}.\n{str(e)[:200]}\n"
            f"Accept the licence at https://huggingface.co/{MODEL_NAME} "
            f"and check https://huggingface.co/settings/gated-repos")
else:
    print(f"{MODEL_KEY} is ungated — no login needed")


indicbart_pre is ungated — no login needed


In [6]:
# ============================================================
# CELL 3 — load the dataset
# ------------------------------------------------------------
# Patches applied automatically when the raw CSV is used
# instead of the output of make_test_splits.py:
#   • article_headline → headline  (column rename)
#   • synthetic row_key added       (mai_0000, mai_0001, …)
#   • EXPECT_N set to actual row count
# ============================================================
df = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)

# --- patch: rename article_headline → headline if needed ---
if "article_headline" in df.columns and "headline" not in df.columns:
    df.rename(columns={"article_headline": "headline"}, inplace=True)
    print("PATCHED: article_headline → headline")

# --- patch: add row_key if missing ---
if "row_key" not in df.columns:
    df.insert(0, "row_key", [f"mai_{i:04d}" for i in range(len(df))])
    print("PATCHED: synthetic row_key added")

# --- set EXPECT_N to actual row count ---
EXPECT_N = len(df)

TEXT_COL, REF_COL, GEN_COL = "article_text", "headline", "generated"
for c in ("row_key", TEXT_COL, REF_COL):
    assert c in df.columns, f"{TEST_CSV} lacks '{c}'"
assert df["row_key"].is_unique, "duplicate row_key"

df[GEN_COL] = ""
print(f"{len(df)} test articles loaded | reference = '{REF_COL}'")
print("\ngold headline word counts:",
      df[REF_COL].str.split().str.len().describe()[["mean","50%","min","max"]].round(2).to_dict())


201 test articles loaded | reference = 'headline'

gold headline word counts: {'mean': 9.27, '50%': 9.0, 'min': 1.0, 'max': 17.0}


In [7]:
# ============================================================
# CELL 4 — article cleaning (unchanged from your notebooks)
# ============================================================
_ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u200e\u200f\ufeff"), None)

def clean_article(text):
    if text is None: return ""
    s = str(text)
    if s.strip().lower() in {"nan", "none", "null"}: return ""
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\t\n ")
    return re.sub(r"\s+", " ", s).strip()

df["_clean"] = df[TEXT_COL].map(clean_article)
n_empty_src = int((df["_clean"] == "").sum())
print("empty articles after cleaning:", n_empty_src)


empty articles after cleaning: 3


In [8]:
# ============================================================
# CELL 5 — PROMPT  (verbatim from your one-shot notebooks, exemplar removed)
#   sha1 of SYSTEM_PROMPT + instruction block in the originals: a5f15dc9dc4fe60e
#   identical across Airavata / Sarvam-1 / OpenHathi / Llama-3.1 / Gemma-2
# ============================================================
SYSTEM_PROMPT = (
    "अहाँ एकटा मैथिली समाचार शीर्षक लेखक छी। "
    "अहाँ हमेशा शुद्ध मैथिली मे लिखब, हिंदी मे कहियो नहि लिखब। "
    "अहाँ लेख केर मूल भाव आ भावना केँ 8-10 शब्द केर शीर्षक मे पकड़ब।"
)

def _instruction_block():
    return (
        "नीचाँ देल गेल मैथिली समाचार लेख केर मैथिली भाषा मे 8 सँ 10 शब्द केर "
        "एक शीर्षक लिखू। शीर्षक खाली एक लाइन केर होअय — कैक वाक्य नहि लिखू।\n\n"
        "शीर्षक खाली शुद्ध मैथिली मे होअय केर चाही — हिंदी मे नहि लिखू। "
        "मैथिली केर शब्द इस्तेमाल करू जेना: अछि, छथि, छल, गेल, कयलनि, रहल, "
        "होअय, चाही, अहाँ, हमरा, ओकर, एकर, सँ, केँ, मे।\n\n"
        "जरूरी बात:\n"
        "- शीर्षक सिर्फ 8 सँ 10 शब्द केर होअय। 10 शब्द सँ बेसी नहि लिखू।\n"
        "- खाली एक लाइन लिखू। दोसर वाक्य नहि जोड़ू।\n"
        "- लेख केर भाव आ भावना (खुशी, दुख, चिंता, गुस्सा, आशा, निराशा, "
        "उत्साह, डर, गर्व) केँ शीर्षक मे जरूर झलकय केर चाही। "
        "अगर लेख मे दुख केर बात अछि त शीर्षक मे सेहो ऊ भावना आबय। "
        "अगर खुशी केर बात अछि त शीर्षक मे सेहो खुशी झलकय।\n"
        "- शीर्षक मे लेख केर सभ सँ जरूरी बात आबय।\n"
        "- मुख्य नाम, जगह वा संख्या शीर्षक मे राखू।\n"
        "- कोनो 'शीर्षक:' लेबल, कोट, बुलेट वा नंबर नहि जोड़ू।\n"
        "- अपन तरफ सँ नव जानकारी नहि जोड़ू।"
    )

def build_user_prompt(cleaned):
    """ZERO-SHOT: instruction + article. The exemplar block is deliberately absent."""
    return (_instruction_block()
            + f"\n\nलेख:\n{cleaned}\n\n"
            + "मैथिली शीर्षक (8-10 शब्द, एक लाइन, खाली मैथिली मे):")

# ---- model-specific chat formatting (the ONLY per-model difference) ----
def format_input(cleaned):
    user = build_user_prompt(cleaned)
    if FMT == "airavata":            # <|system|> / <|user|> / <|assistant|>
        return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{user}\n<|assistant|>\n"
    if FMT == "plain":               # base LMs: no chat template
        return f"{SYSTEM_PROMPT}\n\n{user}"
    if FMT == "chat_sys":            # Llama-3.1: system role supported
        return tokenizer.apply_chat_template(
            [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "chat_nosys":          # Gemma-2: no system role -> fold into user
        return tokenizer.apply_chat_template(
            [{"role":"user","content":SYSTEM_PROMPT + "\n\n" + user}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "none":                # IndicBART: language tag only, no instruction
        return f"<2hi> {cleaned} </s>"
    raise ValueError(FMT)

import hashlib
_h = hashlib.sha1((SYSTEM_PROMPT + _instruction_block()).encode()).hexdigest()[:16]
print(f"prompt sha1 = {_h}   (originals: a5f15dc9dc4fe60e)")
print("SHOTS =", SHOTS, "-> no exemplar appended")


prompt sha1 = a5f15dc9dc4fe60e   (originals: a5f15dc9dc4fe60e)
SHOTS = 0 -> no exemplar appended


In [9]:
# ============================================================
# CELL 6 — load model + tokenizer
# ============================================================
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          MBartForConditionalGeneration, BitsAndBytesConfig)

if globals().get("_loaded") == MODEL_NAME:
    print("already loaded:", MODEL_NAME)
else:
    for v in ("model", "tokenizer"):
        if v in globals(): del globals()[v]
    gc.collect(); torch.cuda.empty_cache()

    print("tokenizer:", MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, use_fast=True,
        do_lower_case=False if ARCH == "seq2seq" else None)

    if ARCH == "seq2seq":
        model = MBartForConditionalGeneration.from_pretrained(
            MODEL_NAME, torch_dtype=torch.float32).cuda()
    else:
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"    # required for batched decoder-only generation
        qcfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True) if USE_4BIT else None
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=qcfg, device_map="auto",
            torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
            attn_implementation="eager")

    model.eval(); _loaded = MODEL_NAME
    print("loaded |", f"{torch.cuda.memory_allocated()/1e9:.1f} GB allocated")


tokenizer: ai4bharat/IndicBART


Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

loaded | 1.0 GB allocated


In [10]:
# ============================================================
# CELL 7 — post-processing (NO LENGTH CAP) + generation
# ------------------------------------------------------------
# Removed vs your one-shot notebooks: first-line-only, first-sentence-only,
# and the "cap at 10 words if >12" rule. Those applied to baselines but not to
# fine-tuned IndicBART and produced most of Table 9's length asymmetry.
# ============================================================
_MARKERS = ("<|assistant|>","<|user|>","<|system|>","<|end|>","<|eot_id|>",
            "<|start_header_id|>","<|end_header_id|>","</s>","<s>","[/INST]",
            "<end_of_turn>","<start_of_turn>","<2hi>")

def postprocess(text):
    if not text: return ""
    s = str(text)
    for m in _MARKERS: s = s.replace(m, " ")
    s = re.sub(r'^\s*(मैथिली\s*)?(शीर्षक|सारांश|headline|title)\s*[:：\-]\s*', '', s, flags=re.I)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.strip(' "\u201c\u201d\u2018\u2019')

@torch.no_grad()
def generate_batch(cleans):
    texts = [format_input(c) for c in cleans]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_INPUT_TOKENS,
                    add_special_tokens=(ARCH != "seq2seq")).to(model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS,
              length_penalty=LENGTH_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM,
              do_sample=DO_SAMPLE, early_stopping=True)
    if ARCH == "seq2seq":
        kw["decoder_start_token_id"] = tokenizer.convert_tokens_to_ids("<2hi>")
        out = model.generate(**enc, **kw)
        dec = tokenizer.batch_decode(out, skip_special_tokens=True)
    else:
        kw["pad_token_id"] = tokenizer.pad_token_id
        out = model.generate(**enc, **kw)
        dec = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                     skip_special_tokens=True)
    return [postprocess(t) for t in dec]

print("smoke test on 2 articles:")
for g in generate_batch(df["_clean"].head(2).tolist()):
    print("  ->", g[:110])


smoke test on 2 articles:
  -> , १०० वर्षक बाद मिथिलाक कि होयत देखू, आब अपना सभक बाल बच्चा विश्व परिवेश संग डेग बढा रहल अछि। आब ओ बाबा आदम वल
  -> प्रवीण नारायण चौधरी झा चन्दन केर ओ प्रसिद्ध कार्टून जाहि मे निजभाषा ‘मैथिली’ केर सिनेमा छोड़ि अश्लील आ फूहरता 


In [11]:
# ============================================================
# CELL 8 — generation loop (fresh start, checkpointed, OOM-safe)
# ============================================================
try:    from tqdm.auto import tqdm
except Exception: tqdm = lambda x, **k: x

if os.path.exists(CHECKPOINT_CSV):
    if FRESH_START:
        bak = f"{CHECKPOINT_CSV}.bak.{int(time.time())}"
        os.rename(CHECKPOINT_CSV, bak)
        print("FRESH_START -> old checkpoint moved to", bak)
    else:
        ck = pd.read_csv(CHECKPOINT_CSV, dtype=str, keep_default_na=False)
        m  = dict(zip(ck["row_key"], ck[GEN_COL]))
        df[GEN_COL] = df["row_key"].map(lambda k: m.get(k, ""))
        print("resumed:", int((df[GEN_COL].str.strip() != "").sum()), "rows")

todo = df.index[df[GEN_COL].str.strip() == ""].tolist()
print(f"generating {len(todo)} of {len(df)} rows")

t0 = time.time(); pbar = tqdm(total=len(todo), desc=RUN_TAG, unit="row")
for s in range(0, len(todo), BATCH_SIZE):
    idx   = todo[s:s+BATCH_SIZE]
    batch = df.loc[idx, "_clean"].tolist()
    try:
        outs = generate_batch(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        outs = []
        for a in batch:
            try:              outs.append(generate_batch([a])[0])
            except Exception: outs.append("")
    except Exception as e:
        print(f"[batch {s} failed: {type(e).__name__}] retrying singly")
        outs = []
        for a in batch:
            try:              outs.append(generate_batch([a])[0])
            except Exception: outs.append("")
    for i, o in zip(idx, outs):
        df.at[i, GEN_COL] = o
    df[["row_key", GEN_COL]].to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")
    pbar.update(len(idx))
pbar.close()

n_empty = int((df[GEN_COL].str.strip() == "").sum())
print(f"\nfinished in {(time.time()-t0)/60:.1f} min | empty outputs {n_empty}/{len(df)}")
df.drop(columns=["_clean"]).to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
print("saved ->", FINAL_CSV)
print("\n--- 5 samples ---")
for i in df.index[:5]:
    print(f"GOLD: {df.at[i, REF_COL][:95]}")
    print(f"GEN : {df.at[i, GEN_COL][:95]}\n")


generating 201 of 201 rows


indicbart_pre_mai_headline_zeroshot:   0%|          | 0/201 [00:00<?, ?row/s]


finished in 0.6 min | empty outputs 3/201
saved -> gen_indicbart_pre_mai_headline_zeroshot.csv

--- 5 samples ---
GOLD: भविष्यक गणनाः १०० वर्षक बाद मिथिलाक कि होयत
GEN : , १०० वर्षक बाद मिथिलाक कि होयत देखू, आब अपना सभक बाल बच्चा विश्व परिवेश संग डेग बढा रहल अछि। आ

GOLD: अपना केँ जराउ, पड़ोसी केँ बचाउ
GEN : प्रवीण नारायण चौधरी झा चन्दन केर ओ प्रसिद्ध कार्टून जाहि मे निजभाषा ‘मैथिली’ केर सिनेमा छोड़ि अ

GOLD: “प्रारम्भिक राजतंत्रः विदेह” (मिथिलाक इतिहास, लेखकः डा. उपेन्द्र ठाकुर)
GEN : एहि इतिहास – डा. उपेन्द्र ठाकुर “प्रारम्भिक राजतंत्रः विदेह” (मिथिलाक इतिहास, लेखकः डा.उपेन्द्र

GOLD: मिथिलाराज्य सँ पहिने आर्थिक उपेक्षा तुरन्त प्रभाव सँ दूर हो, डिवलपमेन्ट काउन्सिलक मांग तेज
GEN : ৰककक राज्य निर्माण मे अछिललल। जनवरी १०, २०१८. मैथिली जिन्दाबाद!! मिथिला राज्य निर्माण सेनाक जमी

GOLD: अहाँक याद बड तड़पाबैत रहैयऽ – मैथिली गजल
GEN : गजल साच्चे आँहाक याद बड तड्पाबैत रहैय ! बिछोडक पीडा बहुते सताबैत रखैय !! तरेगन गनि गनि राति कटै



In [17]:
# ============================================================
# CELL 9 — STANDARDIZED EVALUATION  (no regeneration)
# ------------------------------------------------------------
# Loads existing generations — from memory if the kernel is alive, otherwise
# from gen_*.csv / ckpt_*.csv on disk. Nothing is regenerated.
#
# FIX: bert-score crashes on EMPTY strings with newer transformers
#   (BertTokenizer has no attribute build_inputs_with_special_tokens).
#   Empty outputs are now scored 0.0 on every metric WITHOUT being sent to
#   BERTScore. They are still counted in n, never dropped.
# ============================================================
import os, json, glob, hashlib
import numpy as np, pandas as pd

# ---- 1. recover config if the kernel was restarted ----
_g = globals()
LANG      = _g.get("LANG", "mai")
TASK      = _g.get("TASK", "summary")
MODEL_KEY = _g.get("MODEL_KEY", "indicbart_pre")
SEED      = _g.get("SEED", 42)
MAX_NEW_TOKENS = _g.get("MAX_NEW_TOKENS", 200)
REF_COL   = _g.get("REF_COL", "summary")
GEN_COL   = _g.get("GEN_COL", "generated")
MODEL_NAME = _g.get("MODEL_NAME", MODEL_KEY)
SHOTS     = _g.get("SHOTS", 0)
RUN_TAG   = _g.get("RUN_TAG", f"{MODEL_KEY}_{LANG}_{TASK}_zeroshot")
TEST_CSV  = _g.get("TEST_CSV", f"{LANG}_test.csv")
PERITEM_CSV  = f"per_item_{RUN_TAG}.csv"
METRICS_JSON = f"metrics_{RUN_TAG}.json"
NTOK = _g.get("NTOK", {})

# ---- 2. get generations WITHOUT regenerating ----
if "df" in _g and GEN_COL in df.columns and (df[GEN_COL].astype(str).str.strip() != "").any():
    ev = df.copy()
    print("using generations already in memory")
else:
    ref = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)
    if "row_key" not in ref.columns:
        ref["row_key"] = ref["article_text"].astype(str).str.strip().map(
            lambda t: hashlib.sha1(t.encode()).hexdigest()[:16])
    src = next((p for p in (f"gen_{RUN_TAG}.csv", f"ckpt_{RUN_TAG}.csv") if os.path.exists(p)), None)
    if src is None:
        raise SystemExit(f"No saved generations for {RUN_TAG}. Looked for gen_/ckpt_ CSVs in {os.getcwd()}")
    g = pd.read_csv(src, dtype=str, keep_default_na=False)
    m = dict(zip(g["row_key"], g[GEN_COL]))
    ev = ref.copy()
    ev[GEN_COL] = ev["row_key"].map(lambda k: m.get(k, ""))
    print(f"loaded generations from {src}")

refs = ev[REF_COL].fillna("").astype(str).str.strip().tolist()
hyps = ev[GEN_COL].fillna("").astype(str).str.strip().tolist()
N = len(refs)
empty_idx = [i for i, h in enumerate(hyps) if not h]
print(f"n = {N} | empty generations = {len(empty_idx)}"
      + (f"  (rows {empty_idx[:10]})" if empty_idx else ""))

# ---- 3. ROUGE + BLEU (empty hyp -> 0 naturally) ----
import sacrebleu
from rouge_score import rouge_scorer
class _WS:
    def tokenize(self, t): return str(t).split()
sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"],
                              use_stemmer=False, tokenizer=_WS())
per = [sc.score(r, h) for r, h in zip(refs, hyps)]
r1 = [p["rouge1"].fmeasure for p in per]
r2 = [p["rouge2"].fmeasure for p in per]
rl = [p["rougeL"].fmeasure for p in per]
bl = [0.0 if not h else
      sacrebleu.sentence_bleu(h, [r], smooth_method="exp", tokenize="intl").score / 100
      for h, r in zip(hyps, refs)]
cb = sacrebleu.corpus_bleu(hyps, [refs], tokenize="intl")

# ---- 4. BERTScore on NON-EMPTY pairs only; empties get 0.0 ----
from bert_score import score as bert_score_fn
ok = [i for i in range(N) if hyps[i] and refs[i]]
bs = [0.0] * N
if ok:
    _, _, F = bert_score_fn([hyps[i] for i in ok], [refs[i] for i in ok],
                            lang="hi", model_type="bert-base-multilingual-cased",
                            batch_size=8, verbose=False)
    for j, i in enumerate(ok):
        bs[i] = float(F[j])
print(f"BERTScore computed on {len(ok)}/{N}; {N-len(ok)} empty scored 0.0")

# ---- 5. aggregate with bootstrap CIs ----
def ci(v, n=1000):
    rng = np.random.default_rng(SEED); a = np.asarray(v, float)
    b = [a[rng.integers(0, len(a), len(a))].mean() for _ in range(n)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]

lens = [len(h.split()) for h in hyps]
ntok = [NTOK.get(k, np.nan) for k in ev["row_key"]] if NTOK else [np.nan] * N
hit_cap = (float(np.nanmean([n >= MAX_NEW_TOKENS - 1 for n in ntok]))
           if NTOK else None)

M = {"run": RUN_TAG, "model": MODEL_NAME, "lang": LANG, "task": TASK, "shots": SHOTS,
     "n": N, "n_empty": len(empty_idx),
     "valid_rate": float(1 - len(empty_idx) / N),
     "bleu_sentence": float(np.mean(bl)), "bleu_sentence_ci": ci(bl),
     "rouge1": float(np.mean(r1)), "rouge1_ci": ci(r1),
     "rouge2": float(np.mean(r2)), "rouge2_ci": ci(r2),
     "rougeL": float(np.mean(rl)), "rougeL_ci": ci(rl),
     "bertscore_f1": float(np.mean(bs)), "bertscore_f1_ci": ci(bs),
     "bertscore_f1_nonempty": float(np.mean([bs[i] for i in ok])) if ok else 0.0,
     "brevity_penalty": float(cb.bp), "bleu_corpus": float(cb.score / 100),
     "len_mean": float(np.mean(lens)), "len_median": float(np.median(lens)),
     "len_std": float(np.std(lens)),
     "pct_80_100w": float(np.mean([80 <= l <= 100 for l in lens])),
     "pct_hit_token_cap": hit_cap}

print(f"\n{'='*64}\n{RUN_TAG}   n={N}   valid={M['valid_rate']:.1%}\n{'='*64}")
for k in ("bleu_sentence", "rouge1", "rouge2", "rougeL", "bertscore_f1"):
    lo, hi = M[k + "_ci"]
    print(f"  {k:<15} {M[k]:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  bertscore (non-empty only) {M['bertscore_f1_nonempty']:.4f}")
print(f"  brevity_penalty {M['brevity_penalty']:.4f}")
print(f"  length mean={M['len_mean']:.1f} med={M['len_median']:.0f} "
      f"in 80-100w={M['pct_80_100w']:.1%}")
if hit_cap is not None:
    print(f"  hit {MAX_NEW_TOKENS}-token cap: {hit_cap:.1%}")
else:
    print("  hit-token-cap: n/a (kernel restarted; token counts were in memory)")

pd.DataFrame({"row_key": ev["row_key"], "ref": refs, "hyp": hyps,
              "rouge1": r1, "rouge2": r2, "rougeL": rl, "bleu": bl,
              "bertscore": bs, "len": lens, "ntok": ntok}
             ).to_csv(PERITEM_CSV, index=False, encoding="utf-8-sig")
json.dump(M, open(METRICS_JSON, "w"), indent=2, ensure_ascii=False)
print(f"\nsaved {PERITEM_CSV}\nsaved {METRICS_JSON}")

if empty_idx:
    print(f"\nEmpty generations at test rows {empty_idx} — report valid_rate "
          f"({M['valid_rate']:.1%}) in the paper; these count as 0, not dropped.")


using generations already in memory
n = 201 | empty generations = 3  (rows [42, 73, 174])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore computed on 198/201; 3 empty scored 0.0

indicbart_pre_mai_headline_zeroshot   n=201   valid=98.5%
  bleu_sentence   0.0351   95% CI [0.0294, 0.0420]
  rouge1          0.1387   95% CI [0.1229, 0.1549]
  rouge2          0.0552   95% CI [0.0442, 0.0668]
  rougeL          0.1135   95% CI [0.1007, 0.1262]
  bertscore_f1    0.6714   95% CI [0.6562, 0.6827]
  bertscore (non-empty only) 0.6815
  brevity_penalty 1.0000
  length mean=38.1 med=39 in 80-100w=0.0%
  hit-token-cap: n/a (kernel restarted; token counts were in memory)

saved per_item_indicbart_pre_mai_headline_zeroshot.csv
saved metrics_indicbart_pre_mai_headline_zeroshot.json

Empty generations at test rows [42, 73, 174] — report valid_rate (98.5%) in the paper; these count as 0, not dropped.


In [1]:
# ============================================================
# CELL 10 — protocol manifest (feeds the Section V-B table) + cleanup
# ============================================================
manifest = {
 "run":RUN_TAG, "model_name":MODEL_NAME, "architecture":ARCH,
 "chat_format":FMT, "language":LANG, "task":TASK,
 "eval_set":TEST_CSV, "n_eval":EXPECT_N, "split_seed":SEED,
 "reference_column":REF_COL,
 "shots":SHOTS, "exemplar":None,
 "prompt_sha1":_h, "affect_clause_in_prompt":True,
 "decoding":{"num_beams":NUM_BEAMS, "length_penalty":LENGTH_PENALTY,
             "no_repeat_ngram_size":NO_REPEAT_NGRAM, "do_sample":DO_SAMPLE,
             "max_new_tokens":MAX_NEW_TOKENS, "max_input_tokens":MAX_INPUT_TOKENS},
 "precision":("float32" if ARCH=="seq2seq" else ("4bit-nf4" if USE_4BIT else "bfloat16")),
 "postprocessing":"marker + label cleanup only; NO length cap",
 "empty_output_policy":"scored as 0, never dropped",
 "metrics":{"rouge":"word-level whitespace, per-instance F1, averaged",
            "bleu":"sentence-level, smooth_method=exp, tokenize=intl, /100",
            "bertscore":"bert-base-multilingual-cased, lang=hi, no rescaling",
            "ci":"1000-sample bootstrap, 95% percentile"},
}
json.dump(manifest, open(MANIFEST_JSON,"w"), indent=2, ensure_ascii=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

for v in ("model","tokenizer"):
    if v in globals(): del globals()[v]
globals().pop("_loaded", None)
gc.collect(); torch.cuda.empty_cache()
print("\nGPU released. Change MODEL_KEY in Cell 1 and Run All for the next model.")


NameError: name 'RUN_TAG' is not defined

## After all six runs

### Build the Maithili headline column of Table 10
```python
import glob, json, pandas as pd
rows = [json.load(open(f)) for f in sorted(glob.glob("metrics_*_mai_headline_zeroshot.json"))]
t = pd.DataFrame(rows)[["model","n","valid_rate","bleu_sentence","rouge1","rouge2",
                        "rougeL","bertscore_f1","brevity_penalty","len_mean","pct_over_10w"]]
print(t.round(3).to_string(index=False))
t.to_csv("table10_mai_headline_zeroshot.csv", index=False)
```

### Re-score fine-tuned IndicBART with the SAME evaluator
Do **not** retrain. Load its saved test predictions into `df[GEN_COL]`, set
`RUN_TAG = "ftindicbart_mai_headline"`, and run Cell 9 only. This removes the last
three metric differences: fine-tuned currently uses `13a` BLEU, corpus aggregation,
and a hand-written ROUGE implementation.

### Test every "A beats B" claim before writing it
```python
import numpy as np, pandas as pd
a = pd.read_csv("per_item_llama31_mai_headline_zeroshot.csv").set_index("row_key")
b = pd.read_csv("per_item_ftindicbart_mai_headline.csv").set_index("row_key")
k = a.index.intersection(b.index)
d = (a.loc[k,"rouge1"] - b.loc[k,"rouge1"]).values
rng = np.random.default_rng(42)
lo, hi = np.percentile([d[rng.integers(0,len(d),len(d))].mean() for _ in range(1000)],
                       [2.5, 97.5])
print(f"mean diff {d.mean():+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
```
An interval containing zero means the paper must report the two as statistically
indistinguishable rather than naming a winner.

### Expect low numbers, and report them as a finding
Zero-shot ROUGE on Maithili will likely be near zero for several models. That is the
answer to research gap 2 in Section I-B, not an embarrassment. Record **how** each model
fails — Hindi output, degenerate repetition, prompt echo, empty — because `valid_rate`
plus the failure mode is a stronger claim than a table of zeros. Sample 20 outputs per
model and characterise it in two sentences.

### Then repeat for the other three cells
`LANG="bho"` for Bhojpuri headline, and `TASK="summary"` with `MAX_NEW_TOKENS=200`
for both summary tasks. The summary prompt differs — paste your validated summary
instruction into Cell 5 when you switch.
